# 基于HuggingFace 的预训练语言模型实践
HuggingFace 是一个开源自然语言处理软件库。其的目标是通过提供一套全面的工具、库和模型，使得自然语言处理技术对开发人员和研究人员更加易于使用。HuggingFace 最著名的贡献之一是Transformer 库，基于此研究人员可以快速部署训练好的模型以及实现新的网络结构。除此之外，HuggingFace 还提供了Dataset 库，可以非常方便地下载自然语言处理研究中最常使用的基准数据集。本节中，将以构建BERT 模型为例，介绍基于Huggingface 的BERT 模型构建和使用方法。  

In [ ]:
import logging
import numpy as np
import math
import os
import sys
from dataclasses import dataclass, field
from itertools import chain
from typing import Optional, List, Dict, Any, Mapping
from pathlib import Path
import datasets
import torch
from datasets import load_dataset, concatenate_datasets

import transformers
from transformers import (
    CONFIG_MAPPING,
    MODEL_FOR_CAUSAL_LM_MAPPING,
    AutoConfig,
    AutoModelForCausalLM,
    LlamaForCausalLM,
    LlamaTokenizer,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    is_torch_tpu_available,
    set_seed,
)
from transformers.testing_utils import CaptureLogger
from transformers.trainer_utils import get_last_checkpoint
from transformers.utils import send_example_telemetry
from transformers.utils.versions import require_version

from sklearn.metrics import accuracy_score
from peft import LoraConfig, TaskType, get_peft_model, PeftModel, get_peft_model_state_dict
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR

In [ ]:
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizerFast
from transformers import BertForMaskedLM, BertConfig
from transformers import DataCollatorForLanguageModeling

## 1. 数据集合准备
常见的用于预训练语言模型的大规模数据集都可以在Dataset 库中直接下载并加载。例如，如果使用维基百科的英文语料集合，可以直接通过如下代码完成数据获取：

In [ ]:
from datasets import concatenate_datasets,load_dataset
bookcorpus= load_dataset("bookcorpus",split="train")
wiki =load_dataset("wikipedia","20230601.en", split="train")
#仅保留'text'列
wiki =wiki.remove_columns([col for col in wiki.column_names if col!= "text"])
dataset =concatenate_datasets([bookcorpus,wiki])
#将数据集合切分为90%用于训练，10%用于测试
d= dataset.train_test_split(test_size=0.1)


**将数据集合切分为90% 用于训练，10% 用于测试d $=$ dataset.train_test_split(test_size=0.1)**

接下来将训练和测试数据分别保存在本地文件中，代码如下所示：  

In [ ]:
def dataset_to_text(dataset,output_filename="data.txt"):
    """将数据集文本保存到磁盘的通用函数"""
    with open(output_filename, "w") as f:
        for t in dataset["text"]:
            print(t, file=f)

#将训练集保存为train.txt
dataset_to_text(d["train"], "train.txt")
#将测试集保存为test.txt
dataset_to_text(d["test"], "test.txt")

## 2. 训练词元分析器（Tokenizer）  

BERT 采用了WordPiece 分词，根据训练语料中的词频决定是否将一个完整的词切分为多个词元。因此，需要首先训练词元分析器（Tokenizer）。可以使用transformers 库中的BertWordPiece-Tokenizer 类来完成任务，代码如下所示：  

In [ ]:
special_tokens =[
    "[PAD]","[UNK]", "[CLS]","[SEP]","[MASK]","<S>","<T>"
]
#如果要根据训练和测试两个集合训练词元分析器，需要修改files
#files=["train.txt","test.txt"]
#仅根据训练集合训练词元分析器
files =["train.txt"]
#BERT中采用的默认词表大小30522，可以随意修改
vocab_size= 30_522
#最大序列长度，长度越低训练速度越快
max_length= 512

#是否将长样本截断
truncate_longer_samples = False
#初始化WordPiece词元分析器
tokenizer=BertWordPieceTokenizer()
#训练词元分析器
tokenizer.train(files=files,vocab_size=vocab_size, special_tokens=special_tokens)
#允许截断达到最大512词元
tokenizer.enable_truncation(max_length=max_length)
model_path="pretrained-bert"
#如果文件夹不存在，则首先创建文件夹
if not os.path.isdir(model_path):
    os.mkdir(model_path)
#保存词元分析器模型
tokenizer.save_model(model_path)
#将一些词元分析器中的配置保存到配置文件，包括特殊词元，转换为小写，最大序列长度等。
with open(os.path.join(model_path,"config.json"),"w") as f:
    tokenizer_cfg ={
        "do_lower_case": True,
        "unk_token":"[UNK]",
        "sep_token":"[SEP]",
        "pad_token":"[PAD]",
        "cls_token":"[CLS]",
        "mask_token":"[MASK]",
        "model_max_length":max_length,
        "max_len":max_length,
    }
json.dump(tokenizer_cfg,f)
#当词元分析器进行训练和配置时，将其装载到BertTokenizerFast
tokenizer=BertTokenizerFast.from_pretrained(model_path)

## 3. 预处理语料集合  

在启动整个模型训练之前，还需要将预训练语料根据训练好的Tokenizer 进行处理。如果文档长度超过512 个词元（Token），那么就直接进行截断。数据处理代码如下所示：  

In [ ]:
def encode_with_truncation(examples):
    """使用词元分析对句子进行处理并截断的映射函数（Mappingfunction）"""
    return tokenizer(examples["text"],truncation=True,padding="max_length",
        max_length=max_length,return_special_tokens_mask=True)

def encode_without_truncation(examples):
    """使用词元分析对句子进行处理但是不截断的映射函数（Mappingfunction）"""
    return tokenizer(examples["text"],return_special_tokens_mask=True)
#编码函数将依赖于truncate_longer_samples变量
encode =encode_with_truncation if truncate_longer_samples else encode_without_truncation
#对训练数据集进行分词处理
train_dataset =d["train"].map(encode,batched=True)
#对测试数据集进行分词处理
test_dataset =d["test"].map(encode,batched=True)
if truncate_longer_samples:
    #移除其他列，并将input_ids和attention_mask设置为PyTorch张量
    train_dataset.set_format(type="torch",columns=["input_ids","attention_mask"])
    test_dataset.set_format(type="torch",columns=["input_ids","attention_mask"])
else:
    #移除其他列，将它们保留为Python列表
    test_dataset.set_format(columns=["input_ids","attention_mask", "special_tokens_mask"])
    train_dataset.set_format(columns=["input_ids","attention_mask","special_tokens_mask"])

`truncate_longer_samples` 布尔变量来控制用于对数据集进行词元处理的`encode()` 回调函数。如果设置为`True`，则会截断超过最大序列长度（`max_length`）的句子。否则，不会截断。如果设为`truncate_longer_samples` 为`False`，需要将没有截断的样本连接起来，并组合成固定长度的向量。  

In [ ]:
from itertools import chain
#主要数据处理函数，拼接数据集中的所有文本并生成最大序列长度的块
def group_texts(examples):
    #拼接所有文本
    concatenated_examples= {k:list(chain(*examples[k])) for k in examples.keys()}
    total_length= len(concatenated_examples[list(examples.keys())[0]])
    #舍弃了剩余部分，如果模型支持填充而不是舍弃，你可以根据需要自定义这部分
    if total_length >=max_length:
        total_length =(total_length //max_length) *max_length
    #按照最大长度分割成块
    result= {
        k:[t[i:i +max_length] for i in range(0, total_length,max_length)]
        for k, t in concatenated_examples.items()
    }
    return result
#请注意，使用batched=True，此映射一次处理1,000个文本
#因此group_texts会为这1,000个文本组抛弃不足的部分。
#可以在这里调整batch_size，但较高的值可能会使预处理速度较慢
#
#为了加速这一部分，我们使用了多进程处理。
#请查看map方法的文档以获取更多信息
#https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasets.Dataset.map
if not truncate_longer_samples:
    train_dataset =train_dataset.map(group_texts,batched=True,
        desc=f"Groupingtexts in chunksof {max_length}")
    test_dataset =test_dataset.map(group_texts, batched=True,
        desc=f"Grouping textsinchunksof {max_length}")
    #将它们从列表转换为PyTorch张量
    train_dataset.set_format("torch")
    test_dataset.set_format("torch")

## 4. 模型训练  
在构建了处理好的预训练语料之后，就可以开始模型训练。代码如下所示：  

In [ ]:
#使用配置文件初始化模型
model_config= BertConfig(vocab_size=vocab_size,max_position_embeddings=max_length)
model =BertForMaskedLM(config=model_config)
#初始化数据整理器，随机屏蔽20%（默认为15%）的标记，
#用于遮盖语言建模（MLM）任务。
data_collator =DataCollatorForLanguageModeling(
    tokenizer=tokenizer,mlm=True,mlm_probability=0.2
)
training_args =TrainingArguments(
    output_dir=model_path, #输出目录，用于保存模型检查点
    evaluation_strategy="steps", #每隔`logging_steps`步进行评估。
    overwrite_output_dir=True,
    num_train_epochs=10, #训练时的轮数，可以根据需要进行调整
    per_device_train_batch_size=10, #训练批量大小，请根据您的GPU内存容量将其设置得尽可能大
    gradient_accumulation_steps=8, #在更新权重之前累积梯度
    per_device_eval_batch_size=64, #评估批量大小
    logging_steps=1000, #每隔1000步进行评估，记录并保存模型检查点
    save_steps=1000,
    #load_best_model_at_end=True, #是否在训练结束时加载最佳模型（根据损失）
    #save_total_limit=3, #如果磁盘空间有限，您可以限制只保存3个模型权重
)
trainer =Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)
#训练模型
trainer.train()

训练开始后，可以得到如下输出结果：  

```Bash
[10135/79670 18:53:08 $<$ 129:35:53, 0.15 it/s, Epoch 1.27/10]   
Step Training Loss Validation Loss   
1000 6.904000 6.558231   
2000 6.498800 6.401168   
3000 6.362600 6.277831   
4000 6.251000 6.172856   
5000 6.155800 6.071129   
6000 6.052800 5.942584   
7000 5.834900 5.546123  

8000 5.537200 5.248503   
9000 5.272700 4.934949   
10000 4.915900 4.549236  
```

## 5. 模型使用  

基于训练好的模型，可以针对不同应用需求进行使用，以句子补全为例的代码如下所示：

In [ ]:
#加载模型检查点
model =BertForMaskedLM.from_pretrained(os.path.join(model_path,"checkpoint-10000"))
#加载分词器
tokenizer= BertTokenizerFast.from_pretrained(model_path)
fill_mask= pipeline("fill-mask",model=model,tokenizer=tokenizer)
#进行预测
examples=[
    "Today's mosttrendinghashtagson[MASK]is DonaldTrump",
    "The[MASK]wascloudyyesterday,buttodayit's rainy.",
]
for example in examples:
    for prediction in fill_mask(example):
        print(f"{prediction['sequence']},confidence: {prediction['score']}")
    print("="*50)

通过上述代码可以得到如下输出：  

```Bash
today'smost trending hashtagsontwitterisdonaldtrump, confidence:0.1027069091796875
today'smost trending hashtagsonmondayisdonald trump, confidence:0.09271949529647827
today'smost trending hashtagsontuesdayisdonaldtrump, confidence:0.08099588006734848
today'smost trending hashtagsonfacebookisdonaldtrump,confidence:0.04266013577580452
today'smost trending hashtagsonwednesdayis donald trump,confidence:0.04120611026883125
==================================================
the weatherwascloudyyesterday,buttodayit'srainy.,confidence:0.04445931687951088
the daywascloudyyesterday, buttodayit'srainy.,confidence:0.037249673157930374
the morningwascloudyyesterday,buttodayit'srainy.,confidence:0.023775646463036537
the weekendwascloudyyesterday,buttodayit'srainy.,confidence:0.022554103285074234
the storm was cloudyyesterday, buttodayit's rainy., confidence:0.019406016916036606
==================================================
```